## Read the data

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

APPROACH = "DECOY_PATHS"
# APPROACH = "TOTAL_PATHS"

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = list(sorted(get_available_alleles(DATA_PATH)))

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'
          ]
ALLELES = [
    'HLA-A*01:01',
    'HLA-A*03:01',
    # 'HLA-A*02:01б'
   'HLA-DRB1*01:01'

          ]
selected_alleles = ALLELES
ALLELES

['HLA-A*01:01', 'HLA-A*03:01', 'HLA-DRB1*01:01']

## 2.0 Merge alleles together

In [10]:
# combination_size = len(ALLELES)
ALLELES = sorted(['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'])
# ALLELES = sorted([ 'HLA-A*03:01'])

ALLELES = sorted(['HLA-DRB1*01:01'])


combination_size = 1
TOTAL_SPLITS = 5

In [11]:
import itertools
from collections import defaultdict
per_allele_per_kfold_per_length_binders_train_new = dict()

per_allele_per_kfold_per_length_binders_test_new = dict()

per_kfold_per_length_per_allele_counts_old_train = dict()
per_kfold_per_length_per_allele_counts_old_test = dict()





OLD_ALLELES = ALLELES
NEW_ALELLES = list()
selected_lengths = np.arange(8,20)

# initialize count dicts
for allele_name in OLD_ALLELES:
    for split_num in range(TOTAL_SPLITS):
        per_kfold_per_length_per_allele_counts_old_train[split_num] = defaultdict(int)
        per_kfold_per_length_per_allele_counts_old_test[split_num] = defaultdict(int)





for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()


    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()

        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        # #calculate allele_weights here
        # # allele weight is   len(allele data)/ total_data_length
        # for old_allele in combination:
        #     lengths_list = {
        #         length: len(length_binders)
        #         for length, length_binders
        #         in per_allele_per_kfold_per_length_binders_train[old_allele][split_num].values()
        #     }


        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            if target_length not in per_kfold_per_length_per_allele_counts_old_train[split_num]:
                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length] = dict()
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length] = dict()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]

                per_kfold_per_length_per_allele_counts_old_train[split_num][target_length][old_allele] = len(single_allele_data_array_train)
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)
                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                per_kfold_per_length_per_allele_counts_old_test[split_num][target_length][old_allele] = len(single_allele_data_array_test)
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            if split_num == 0:
                print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)


per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-DRB1*01:01
Length 8: HLA-DRB1*01:01 - 39 
Length 9: HLA-DRB1*01:01 - 113 
Length 10: HLA-DRB1*01:01 - 123 
Length 11: HLA-DRB1*01:01 - 175 
Length 12: HLA-DRB1*01:01 - 368 
Length 13: HLA-DRB1*01:01 - 826 
Length 14: HLA-DRB1*01:01 - 1184 
Length 15: HLA-DRB1*01:01 - 6886 
Length 16: HLA-DRB1*01:01 - 1413 
Length 17: HLA-DRB1*01:01 - 984 
Length 18: HLA-DRB1*01:01 - 708 
Length 19: HLA-DRB1*01:01 - 316 
Length 20: HLA-DRB1*01:01 - 505 
Length 21: HLA-DRB1*01:01 - 177 
Length 22: HLA-DRB1*01:01 - 80 
Length 23: HLA-DRB1*01:01 - 57 
Length 24: HLA-DRB1*01:01 - 67 
Length 25: HLA-DRB1*01:01 - 50 
Length 26: HLA-DRB1*01:01 - 15 
Length 27: HLA-DRB1*01:01 - 12 
Length 28: HLA-DRB1*01:01 - 8 
Length 30: HLA-DRB1*01:01 - 7 


## Calculate weights for data

In [12]:

def extract_count_df(per_kfold_per_length_per_allele_counts):
    result_dict = {}
    for kfold,  per_length_per_allele_counts in per_kfold_per_length_per_allele_counts.items():
        for length, per_allele_counts in per_length_per_allele_counts.items():
            for allele_name , count in  per_allele_counts.items():
                result_dict[(allele_name, kfold, length)] =  {'cnt': count }
    count_df = pd.DataFrame.from_dict(result_dict, orient="index")
    count_df.index.names = ['allele', 'split', 'length']
    # count_df = count_df.groupby(['allele', 'length'])['cnt'].mean()
    return count_df



count_df_train = extract_count_df(per_kfold_per_length_per_allele_counts_old_train)
count_df_test = extract_count_df(per_kfold_per_length_per_allele_counts_old_test)


In [13]:
count_df_train.index.levels

FrozenList([['HLA-DRB1*01:01'], [0, 1, 2, 3, 4], [8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30]])

In [16]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model

TARGET_LENGTHS= [13, 14, 15, 16, 17, 18]
# TARGET_LENGTHS= [8, 9, 10, 11, 12]
NUM_RUNS = 2
TOTAL_SPLITS = 5
USE_WEIGHTS = True

### Define weights for data

In [17]:
per_allele_per_kfold_per_length_weights = dict()


for combined_allele_name in ALLELES:
    per_allele_per_kfold_per_length_weights[combined_allele_name] = dict()
    old_alleles = combined_allele_name.split('+')
    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_weights[combined_allele_name][split_num] = dict()
        sub_count_df_train = count_df_train.loc[(count_df_train.index.get_level_values('split') == 0) &
                                                (count_df_train.index.get_level_values('allele').isin(old_alleles)) &
                                                (count_df_train.index.get_level_values('length').isin(TARGET_LENGTHS))]
        total_data_length = sub_count_df_train['cnt'].sum()
        print(f"Total data length for split {split_num} is {total_data_length}")
        for target_length in TARGET_LENGTHS:

            current_weights_list = list()
            weights_dict = dict()
            for old_allele_name in old_alleles:
                old_allele_cnt = count_df_train.loc[(old_allele_name, split_num, target_length)]['cnt']
                current_weights_list.extend([total_data_length/old_allele_cnt if USE_WEIGHTS else 1 for i in range(old_allele_cnt)])
                weights_dict[old_allele_name] = total_data_length/old_allele_cnt if USE_WEIGHTS else 1
            per_allele_per_kfold_per_length_weights[combined_allele_name][split_num][target_length] = current_weights_list
            print(f"Weights for lentgh {target_length} are {weights_dict}")


Total data length for split 0 is 12001
Weights for lentgh 13 are {'HLA-DRB1*01:01': 14.529055690072639}
Weights for lentgh 14 are {'HLA-DRB1*01:01': 10.13597972972973}
Weights for lentgh 15 are {'HLA-DRB1*01:01': 1.7428115015974441}
Weights for lentgh 16 are {'HLA-DRB1*01:01': 8.493276716206653}
Weights for lentgh 17 are {'HLA-DRB1*01:01': 12.196138211382113}
Weights for lentgh 18 are {'HLA-DRB1*01:01': 16.95056497175141}
Total data length for split 1 is 12001
Weights for lentgh 13 are {'HLA-DRB1*01:01': 14.529055690072639}
Weights for lentgh 14 are {'HLA-DRB1*01:01': 10.127426160337553}
Weights for lentgh 15 are {'HLA-DRB1*01:01': 1.7428115015974441}
Weights for lentgh 16 are {'HLA-DRB1*01:01': 8.493276716206653}
Weights for lentgh 17 are {'HLA-DRB1*01:01': 12.196138211382113}
Weights for lentgh 18 are {'HLA-DRB1*01:01': 16.95056497175141}
Total data length for split 2 is 12001
Weights for lentgh 13 are {'HLA-DRB1*01:01': 14.529055690072639}
Weights for lentgh 14 are {'HLA-DRB1*01:01'

# Read models

In [4]:
## PARAMETERS
RUNS = 2
# RUNS = 1
total_splits = 1
SELECTED_RUN = 0
SELECTED_RUN = {0,1,2,3,4,5, 6, 7, 8, 9}
SELECTED_RUN = {0,1}

# SELECTED_RUN = {0}
#
SELECTED_SPLIT = 0

MODELS_DESCRIPTIONS = {
    "single_allele_imbalanced": {
        "path": "experiment_results/alleles_with_groups_single_allele_imbalanced_data/",
        "states": 25,
        "params_string": "5-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
        "target_lengths": [9, 10],
    },
    "single_allele_balanced": {
        "path": "experiment_results/alleles_with_groups_single_allele_balanced_data/",
        "states": 25,
        "params_string": "5-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
        "target_lengths": [9, 10]
    },
    "multiple_allele_balanced": {
        "path": "experiment_results/alleles_with_groups_multiple_allele_balanced_data/",
        "states": 25,
        "params_string": "5-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-A*01:01', 'HLA-A*03:01',
                    # 'HLA-A*02:01'
                   ],
        "target_lengths": [9, 10, 11, 12]
    },
    "single_allele_balanced_class2": {
        "path": "experiment_results/alleles_with_groups_single_allele_balanced_data/",
        "states": 25,
        "params_string": "5-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01'
                    # 'HLA-A*02:01'
                   ],
        "target_lengths": [13, 14, 15, 16, 17, 18]
    },

        "single_allele_balanced_class2_biger": {
        "path": "experiment_results/alleles_with_groups_and_joiner_states_single_allele_balanced_data/",
        "states": 49,
        "params_string": "5-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01'
                    # 'HLA-A*02:01'
                   ],
        "target_lengths": [13, 14, 15, 16, 17, 18]
    },
    "multiple_allele_imbalanced": {
        "path": "experiment_results/alleles_with_groups_multiple_allele_imbalanced_data/",
        "states": 25,
        "params_string": "5-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-A*01:01', 'HLA-A*03:01',
                    # 'HLA-A*02:01'
                   ],
        "target_lengths": [9, 10]
    },

    "multiple_allele_balanced_longer_training": {
        "path": "experiment_results/alleles_with_groups_multiple_allele_balanced_data/",
        "states": 25,
        "params_string": "100-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-A*01:01', 'HLA-A*03:01',
                    # 'HLA-A*02:01'
                   ],
        "target_lengths": [9, 10]
    },
    "mix_of_alleles":
        {
            "path": "experiment_results/alleles_with_groups/",
            "states": 25,
            "params_string": "5-min_iters",
            "alleles": ['HLA-A*01:01', 'HLA-A*03:01'],
            "target_lengths": [9, 10]
        },

    "single_allele_balanced_class2_joiners": {
        "path": "experiment_results/alleles_with_groups_and_joiner_states_single_allele_balanced_data/",
        "states": 49,
        "params_string": "5-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [13, 14, 15, 16, 17, 18]}
    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [5]:
MODELS_DESCRIPTIONS.keys()

dict_keys(['single_allele_imbalanced', 'single_allele_balanced', 'multiple_allele_balanced', 'single_allele_balanced_class2', 'single_allele_balanced_class2_biger', 'multiple_allele_imbalanced', 'multiple_allele_balanced_longer_training', 'mix_of_alleles', 'single_allele_balanced_class2_joiners'])

In [6]:
selected_models = [
                 #'single_allele_imbalanced',
                 #'multiple_allele_imbalanced',
                    # 'single_allele_balanced',
                   # 'multiple_allele_balanced',
    # "single_allele_balanced_class2",
        "single_allele_balanced_class2_joiners"
    # 'multiple_allele_balanced_longer_training'
]


In [7]:
from hmm_logic_methods import load_model

In [8]:
import itertools
import mhcnames

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_binding_model = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    model_alleles = set(model_alleles).intersection(set(ALLELES))
    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        for run_num_b in range(RUNS):
            for split_num_b in range(total_splits):
                #bw
                bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                #bwv
                bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                #v
                v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_binding_model[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [9]:
list(per_name_binding_model.keys())

['0_run-13-14-15-16-17-18-49-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0',
 '0_run-13-14-15-16-17-18-49-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi',
 '0_run-13-14-15-16-17-18-49-st-viterbi-alg-5-min_iters_model-HLA_DRB1_01_01-0',
 '1_run-13-14-15-16-17-18-49-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0',
 '1_run-13-14-15-16-17-18-49-st-baum-welch-alg-5-min_iters_model-HLA_DRB1_01_01-0_plus_viterbi',
 '1_run-13-14-15-16-17-18-49-st-viterbi-alg-5-min_iters_model-HLA_DRB1_01_01-0']

## 2.2 Train
### 2.2.1 Define treshold

In [18]:
stop_threshold = 0
cutoff = 0.0005
per_symbol_portion = -50 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

84.57592592592592

In [28]:
import logomaker
from collections import defaultdict
def probs_to_info_matrix(distribution):
    amino_acids = list(distribution.parameters[0].keys())
    dummy_df = pd.DataFrame(columns = sorted(amino_acids), index=[0],  dtype=float)
    for i, (acid, prob) in enumerate(distribution.parameters[0].items()):
        dummy_df.loc[0, acid] = prob
    dummy_df.index.name = "pos"
    info_matrix = logomaker.transform_matrix(dummy_df, from_type='probability', to_type='information')
    return info_matrix



for model_key, model in per_name_binding_model.items():
    per_state_information = dict()
    per_level_information = defaultdict(float)

    for state in model.states:
        if state not in [model.start, model.end]:
            info_matrix = probs_to_info_matrix(state.distribution)
            per_state_information[state.name] = sum(info_matrix.loc[0])
            # per_level_information[name_to_level[state.name]] += per_state_information[state.name]
    break

In [29]:
df = pd.DataFrame(per_state_information.items(), columns=["state", "info"])

In [30]:
df.sort_values(by='info', ascending=False)

,state,info
35,s035,2.961156
32,s032,2.544958
23,s023,2.204383
37,s037,2.101528
21,s021,2.100292
30,s030,1.818655
16,s016,1.806049
10,s010,1.725221
17,s017,1.657762
40,s040,1.654614
